# Baseline: Random Score Predictor

Samples home and away goals independently from Poisson(λ) where λ is the empirical marginal
goal rate in each fold's training data. True floor — any real model must beat this.

Uses LOTO-CV: 4 folds × 64 matches = 256 pooled evaluation matches. Reports bootstrap 95% CI.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import mlflow
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

SEED = 42
DATA = Path('../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]
N_SAMPLES = 500  # realisations per fold to average out randomness

In [2]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}

In [3]:
# LOTO-CV: for each fold, sample N_SAMPLES realisations and average
fold_results = []
all_pts_per_match = []  # one expected-pts value per eval match (averaged over samples)

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    lh = train['home_score'].mean()
    la = train['away_score'].mean()

    # Average over N_SAMPLES random draws to get stable expected pts per match
    pts_matrix = np.zeros((N_SAMPLES, len(evalf)))
    for i in range(N_SAMPLES):
        rng = np.random.default_rng(SEED + year + i)
        ph = rng.poisson(lh, len(evalf))
        pa = rng.poisson(la, len(evalf))
        pts_matrix[i] = sporza_points_series(
            pd.Series(ph), pd.Series(pa),
            evalf['home_score'].reset_index(drop=True),
            evalf['away_score'].reset_index(drop=True),
        ).values

    # Expected pts per match = mean over samples
    expected_pts = pts_matrix.mean(axis=0)
    all_pts_per_match.extend(expected_pts.tolist())

    fold_mean = expected_pts.mean()
    fold_results.append({'year': year, 'mean_pts': fold_mean, 'lambda_home': lh, 'lambda_away': la})
    print(f'WC {year}: λ_home={lh:.3f} λ_away={la:.3f}  mean_pts={fold_mean:.3f}')

pooled = pd.Series(all_pts_per_match)
ci = bootstrap_ci(pooled)
print(f'\nPooled LOTO-CV ({len(pooled)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')

WC 2010: λ_home=1.507 λ_away=0.968  mean_pts=2.941
WC 2014: λ_home=1.552 λ_away=1.080  mean_pts=2.963


WC 2018: λ_home=1.565 λ_away=1.074  mean_pts=2.893
WC 2022: λ_home=1.572 λ_away=1.071  mean_pts=2.949



Pooled LOTO-CV (256 matches): 2.936 pts/match  95% CI [2.869, 3.004]


In [4]:
mlflow.set_tracking_uri('../mlruns')
mlflow.set_experiment('wk2026-tabular-2026-06-13')

with mlflow.start_run(run_name='random_poisson_loto'):
    mlflow.set_tags({'modality': 'tabular', 'approach': 'random_poisson',
                     'eval_strategy': 'loto_cv', 'dataset_version': 'split_v2'})
    mlflow.log_params({'n_samples': N_SAMPLES, 'seed': SEED})
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
    run_id = mlflow.active_run().info.run_id

print(f'run_id: {run_id}')

run_id: fbe3321e5d1043a182fc5b575a8b33cd
